In [4]:
%load_ext autoreload
%autoreload 2
from sindex.sources.datacite.utils import get_relevant_citations_block_from_ndjson
from sindex.sources.datacite.jobs import (
    batch_slim_datacite_record_to_ndjson_fast,
    batch_find_citations_dc_from_citation_block,
batch_find_citations_dc_from_citation_block_optimized
)
import duckdb
from pathlib import Path
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Slim datacite citations raw ndjson

In [5]:
src_folder = r"I:\pipeline-data\citations\datacite\datacite-raw-with-citations"
dst_folder = r"I:\pipeline-data\citations\datacite\datacite-slim-with-citations"
summary = batch_slim_datacite_record_to_ndjson_fast(
    src_folder = str(src_folder),
    dst_folder = str(dst_folder),
    overwrite = False,          # overwrite for repeatable tests
    accept_gz = True,
    one_line_progress = True
)

# Summary
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\nCompleted.")

Processing 769 files using 40 cores...
[769/769] files completed
Done. files=769 kept=20,147,279 bad=0 time=3665.1s rate≈5,496/rec-per-sec

Summary:
  files_seen: 769
  records_read: 20147279
  records_kept: 20147279
  records_bad_json: 0
  output_dir: \\192.168.20.168\AILarge\pipeline-data\citations\datacite\datacite-slim-with-citations
  elapsed_sec: 3665.14
  rate_rec_per_sec: 5496

Completed.


## Save citations block matching to our DOIs

In [ ]:
get_relevant_citations_block_from_ndjson(
     db_path = r"I:\pipeline-data\records\slim-records\datacite-slim-records.duckdb",
     ndjson_folder = r"I:\pipeline-data\citations\datacite\datacite-slim-with-citations",
     target_table = "my_datasets",
     output_file_path = r"I:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson",
     reset_log = True
 )

[2026-01-24 22:05:22.128294] Batch processing 769 files...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
def count_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        count = 0
        for _ in f:
            count += 1
    return count
file_path = r"I:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
print(f"Total objects: {count_lines(file_path):,}")

## Get Datacite citations

In [2]:
folder_path = r"I:\pipeline-data\citations\datacite\citation_blocks"
out_path = r"I:\pipeline-data\citations\datacite\dc_citations.ndjson"
batch_find_citations_dc_from_citation_block(folder_path, out_path)

Starting processing of 1 files
Processing file 1/1 | Total Citations: 15,168n)

KeyboardInterrupt: 

In [2]:
folder_path = r"I:\pipeline-data\citations\datacite\citation_blocks"
out_path = r"I:\pipeline-data\citations\datacite\dc_citations.ndjson"
oa_db_path = r"I:\pipeline-data\external\openalex-snapshot\duckdb\oa_snapshot.duckdb"

In [ ]:
batch_find_citations_dc_from_citation_block_optimized(folder_path, out_path, oa_db_path)

[*] Found 1 file(s). Connecting to DuckDB...

--- Processing: citation_blocks.ndjson (1/1) ---
[19:22:50] Step 1: Extracting DOIs and loading file into memory...
    -> Extracted 162,150 rows and 82,410 unique citation DOIs.
['10.1063/5.0281079', '10.1186/s12915-025-02229-4', '10.1039/d4dt03495c', '10.1080/01621459.2024.2448857', '10.1039/d5sc05037e', '10.5281/zenodo.12827591', '10.1021/jacs.5c07053', '10.1080/24740527.2024.2425596', '10.1038/s41467-025-63181-z', '10.1038/s41467-025-65670-7']
[19:22:53] Step 2: Querying DuckDB for 82,410 DOIs...


In [3]:
con = duckdb.connect(oa_db_path)

verified_doi = "10.5281/zenodo.12827591" 

# Check Step 1: Is it actually in your set?
print(f"Verified DOI in Set: {verified_doi in unique_dois_in_file}")

# Check Step 2: Querying the DB directly
db_val = conn.execute("SELECT doi FROM openalex_pubdate WHERE doi = ?", [verified_doi]).fetchone()
print(f"Direct DB Lookup: {db_val}")

# Check Step 3: The Join mechanism
# This creates a tiny virtual table with 1 row and tries to join it
test_rel = conn.values([(verified_doi,)]).project("col0 AS search_doi")
join_val = conn.execute("SELECT o.doi FROM test_rel t INNER JOIN openalex_pubdate o ON t.search_doi = o.doi").fetchone()
print(f"Mini-Join Lookup: {join_val}")
con.close()

NameError: name 'unique_dois_in_file' is not defined